In [1]:
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import os

In [2]:
BASE_DIR = Path("/mnt/f/CocDatasets/Perception1")
MODEL_PATH = BASE_DIR / "runs" / "yolov8s_seg_perception_v1" / "weights" / "best.pt"

TEST_DIR = BASE_DIR / "test_images"

print("Model exists:", MODEL_PATH.exists())
print("Test dir exists:", TEST_DIR.exists())

Model exists: True
Test dir exists: True


In [3]:
model = YOLO(str(MODEL_PATH))
print("Model loaded successfully.")

Model loaded successfully.


In [4]:
test_images = sorted([
    p for p in TEST_DIR.iterdir()
    if p.suffix.lower() in [".png", ".jpg", ".jpeg"]
])

print(f"Found {len(test_images)} test images.")
test_images[:5]

Found 6 test images.


[PosixPath('/mnt/f/CocDatasets/Perception1/test_images/1.png'),
 PosixPath('/mnt/f/CocDatasets/Perception1/test_images/2.png'),
 PosixPath('/mnt/f/CocDatasets/Perception1/test_images/3.png'),
 PosixPath('/mnt/f/CocDatasets/Perception1/test_images/4.png'),
 PosixPath('/mnt/f/CocDatasets/Perception1/test_images/5.png')]

In [8]:
r = results[0]

print("Detected objects:", len(r.boxes))

for i, cls_id in enumerate(r.boxes.cls.tolist()):
    conf = r.boxes.conf[i].item()
    cls_name = model.names[int(cls_id)]
    print(f"{i+1}. {cls_name} ({conf:.3f})")

Detected objects: 37
1. wizard_tower (0.965)
2. mortar (0.962)
3. x_bow (0.958)
4. air_sweeper (0.956)
5. mortar (0.952)
6. elixer_storage (0.952)
7. canon (0.952)
8. dark_elixer_storage (0.948)
9. gold_storage (0.947)
10. x_bow (0.945)
11. gold_storage (0.945)
12. archer_tower (0.942)
13. air_defense (0.941)
14. gold_storage (0.941)
15. air_defense (0.939)
16. elixer_storage (0.937)
17. wizard_tower (0.935)
18. x_bow (0.934)
19. gold_storage (0.932)
20. air_sweeper (0.928)
21. canon (0.927)
22. archer_tower (0.926)
23. inferno_tower (0.926)
24. archer_tower (0.924)
25. wizard_tower (0.921)
26. town_hall (0.920)
27. archer_tower (0.920)
28. air_defense (0.915)
29. eagle_artillery (0.913)
30. archer_tower (0.911)
31. inferno_tower (0.910)
32. archer_tower (0.908)
33. bomb_tower (0.840)
34. archer_tower (0.815)
35. wizard_tower (0.796)
36. bomb_tower (0.504)
37. wizard_tower (0.342)


In [9]:
batch_results = model.predict(
    source=[str(p) for p in test_images],
    conf=0.25,
    save=True,
    imgsz=1024
)

print(f"Processed {len(batch_results)} images.")


0: 480x1024 2 canons, 7 archer_towers, 2 mortars, 3 air_defenses, 5 wizard_towers, 2 air_sweepers, 2 bomb_towers, 3 x_bows, 2 inferno_towers, 1 eagle_artillery, 1 town_hall, 4 gold_storages, 2 elixer_storages, 1 dark_elixer_storage, 37.0ms
1: 480x1024 5 archer_towers, 2 mortars, 4 air_defenses, 5 wizard_towers, 1 air_sweeper, 1 bomb_tower, 2 x_bows, 1 inferno_tower, 1 eagle_artillery, 3 gold_storages, 2 elixer_storages, 37.0ms
2: 480x1024 2 canons, 4 archer_towers, 1 mortar, 2 air_defenses, 3 wizard_towers, 1 bomb_tower, 2 x_bows, 1 inferno_tower, 1 gold_storage, 2 elixer_storages, 1 clan_castle, 37.0ms
3: 480x1024 2 canons, 5 archer_towers, 1 mortar, 2 air_defenses, 3 wizard_towers, 1 air_sweeper, 2 x_bows, 2 inferno_towers, 1 eagle_artillery, 1 gold_storage, 4 elixer_storages, 1 clan_castle, 37.0ms
4: 480x1024 3 canons, 5 archer_towers, 3 mortars, 3 air_defenses, 5 wizard_towers, 2 air_sweepers, 1 bomb_tower, 4 x_bows, 3 inferno_towers, 1 eagle_artillery, 1 town_hall, 3 gold_storage

In [10]:
from collections import Counter

counter = Counter()

for r in batch_results:
    for cls_id in r.boxes.cls.tolist():
        counter[model.names[int(cls_id)]] += 1

print("Detection summary:")
print(dict(counter))

Detection summary:
{'wizard_tower': 21, 'mortar': 10, 'x_bow': 13, 'air_sweeper': 6, 'elixer_storage': 18, 'canon': 13, 'dark_elixer_storage': 3, 'gold_storage': 16, 'archer_tower': 34, 'air_defense': 19, 'inferno_tower': 9, 'town_hall': 3, 'eagle_artillery': 4, 'bomb_tower': 5, 'clan_castle': 4}
